# 1. Environment Setup & Imports
We import all libraries and modules necessary for data handling, clustering, deep learning models, training, evaluation, baseline modeling, and visualization.

In [1]:
import os
import json
import pickle
import math
import re
import random
import gc
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import XLMRobertaModel, XLMRobertaTokenizerFast
from torchcrf import CRF
from sklearn.feature_extraction import DictVectorizer
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support, accuracy_score
from sklearn.metrics import homogeneity_score, completeness_score, v_measure_score, matthews_corrcoef, cohen_kappa_score
from sklearn.linear_model import LogisticRegression
# from hmmlearn.hmm import MultinomialHMM
from hmmlearn.hmm import CategoricalHMM
from tqdm.auto import tqdm

/home/taz/Programming/ResearchPOSTagging/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 2. Global Constants & Configuration Variables
We define structural parameters, tag mapping dictionaries, canonical tag lists, random seed values, and directory paths.

In [2]:
# Device mapping
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

# Tag Normalization Dictionary
TAG_MAP = {
    'NN': 'N', 'NNS': 'N', 'NNP': 'N', 'NNPS': 'N', 'NOUN': 'N', 'PROPN': 'N',
    'PRP': 'Pr', 'PRP$': 'Pr', 'PRON': 'Pr', 'PRONOUN': 'Pr', 'PR': 'Pr',
    'JJ': 'Adj', 'JJR': 'Adj', 'JJS': 'Adj', 'ADJ': 'Adj', 'ADJECTIVE': 'Adj',
    'RB': 'Adv', 'RBR': 'Adv', 'RBS': 'Adv', 'ADV': 'Adv', 'ADVERB': 'Adv',
    'VB': 'V', 'VBD': 'V', 'VBG': 'V', 'VBN': 'V', 'VBP': 'V', 'VBZ': 'V', 'VERB': 'V', 'ROOT VERB': 'V',
    'ADP': 'PREP', 'PREP': 'PREP', 'PREPOSITION': 'PREP',
    'CC': 'CONJ', 'CONJ': 'CONJ', 'SCONJ': 'CONJ', 'CON': 'CONJ',
    'IN': 'INTJ',
    '.': '.', ',': ',', 'PUNCT': '.',
}
CANONICAL_TAGS = ['N', 'Pr', 'Adj', 'Adv', 'V', 'PREP', 'CONJ', 'INTJ', '.']
TAG_TO_ID = {tag: idx for idx, tag in enumerate(CANONICAL_TAGS)}
ID_TO_TAG = {idx: tag for tag, idx in TAG_TO_ID.items()}

# Hyperparameters
SEED = 42
MAX_LENGTH = 128
BATCH_SIZE = 8
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.01
TEACHER_EPOCHS = 15
STUDENT_EPOCHS = 20
PATIENCE = 5
ADAPTIVE_THRESHOLD = 0.95

# Directory & Path setups
DATA_PATH = Path('all.txt')
LEXICON_PATH = Path('POS.csv')
OUTPUTS_DIR = Path('outputs')
OUTPUTS_DIR.mkdir(exist_ok=True)
CHECKPOINTS_DIR = OUTPUTS_DIR / 'checkpoints'
CHECKPOINTS_DIR.mkdir(exist_ok=True)

Using device: cuda


# 3. Utility and Helper Functions
Define reproducibility routines, score computation metrics, and garbage collection mechanisms.

In [3]:
def set_seeds(seed=42):
    """Establishes reproducibility seeds."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def compute_macro_f1(y_true, y_pred):
    """Estimates the overall Macro F1 score across categories."""
    report = classification_report(y_true, y_pred, labels=CANONICAL_TAGS, output_dict=True, zero_division=0)
    macro_f1 = np.mean([report[t]['f1-score'] for t in CANONICAL_TAGS if t in report])
    return macro_f1

def gc_collect():
    """Clears memory cache cleanly to release device limits."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

# 4. Data Loading and Preprocessing Definitions
Functions to load dataset records from text/CSV files and configure vocabulary indexing mappings.

In [4]:
def load_word_tag_file(path):
    """Parses the raw tagged corpus file, cleaning whitespace and trailing slashes."""
    sentences = []
    with path.open('r', encoding='utf-16') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            tokens = line.split()
            sent = []
            for tok in tokens:
                tok = tok.strip('/')
                if '/' not in tok:
                    continue
                word, tag = tok.rsplit('/', 1)
                word = word.strip()
                tag = tag.strip()
                if word and tag:
                    sent.append((word, tag))
            if sent:
                sentences.append(sent)
    return sentences

def load_lexicon(path, tag_map, canonical_tags, tag_to_id):
    """Loads POS.csv only once and maps every word to a normalized canonical tag."""
    lexicon = {}
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or ',' not in line:
                continue
            word, tag = line.split(',', 1)
            word = word.strip()
            tag = tag.strip().upper()
            if word and tag:
                mapped = tag_map.get(tag, tag)
                if mapped in tag_to_id:
                    lexicon[word] = mapped
                else:
                    lexicon[word] = 'N'  # Fallback to noun
    return lexicon

def build_char_vocab(sentences):
    """Constructs character index map for the character CNN layer."""
    counter = Counter()
    for sent in sentences:
        for word, _ in sent:
            counter.update(list(word))
    vocab = {'<pad>': 0, '<unk>': 1}
    for char, freq in counter.items():
        vocab[char] = len(vocab)
    return vocab

def encode_char_sequences(batch, char_vocab_dict, max_word_len=20):
    """Pads and builds character tensor for word characters."""
    batch_size = len(batch)
    seq_len = batch[0]['input_ids'].size(0)
    char_ids = torch.zeros(batch_size, seq_len, max_word_len, dtype=torch.long)
    for i, item in enumerate(batch):
        for j, word in enumerate(item['tokens'][:seq_len]):
            for k, ch in enumerate(word[:max_word_len]):
                char_ids[i, j, k] = char_vocab_dict.get(ch, char_vocab_dict['<unk>'])
    return char_ids

def collate_fn(batch):
    """Collates list of sample dictionaries into torch batches."""
    keys = ['input_ids', 'attention_mask', 'labels']
    collated = {k: torch.stack([item[k] for item in batch]) for k in keys}
    collated['char_ids'] = encode_char_sequences(batch, char_vocab)
    collated['tokens'] = [item['tokens'] for item in batch]
    return collated

# 5. Word Feature Extraction & Induction Definitions
Utility logic extracting morphological, suffix, and contextual features of target words.

In [5]:
def extract_features(word, left_contexts=None, right_contexts=None):
    """Extracts spelling patterns and context surrounding words."""
    lower = word.lower()
    feats = {
        'pref1': lower[:1],
        'pref2': lower[:2],
        'pref3': lower[:3],
        'suf1': lower[-1:],
        'suf2': lower[-2:],
        'suf3': lower[-3:],
        'is_title': word.istitle(),
        'is_upper': word.isupper(),
        'has_digit': any(char.isdigit() for char in word),
        'length': len(word),
    }
    if left_contexts is not None:
        for rank, (ctx_w, _) in enumerate(left_contexts[word].most_common(5)):
            feats[f'left_ctx_{rank}'] = ctx_w
    if right_contexts is not None:
        for rank, (ctx_w, _) in enumerate(right_contexts[word].most_common(5)):
            feats[f'right_ctx_{rank}'] = ctx_w
    return feats

# 6. Dataset Class Definition
Implementation of PyTorch Dataset representing custom sequence labeling structure with token alignment mappings.

In [6]:
class POSDataset(Dataset):
    def __init__(self, sentences, tokenizer, max_length=128):
        self.sentences = sentences
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        sent = self.sentences[idx]
        words = [w for w, _ in sent]
        tags = [t for _, t in sent]

        encoding = self.tokenizer(
            words, is_split_into_words=True,
            return_tensors='pt', padding='max_length',
            truncation=True, max_length=self.max_length
        )

        word_ids = encoding.word_ids(batch_index=0)
        labels = []
        prev_wid = None
        for wid in word_ids:
            if wid is None:
                labels.append(-100)
            elif wid != prev_wid:
                tag = tags[wid] if wid < len(tags) else 'N'
                labels.append(TAG_TO_ID.get(tag, TAG_TO_ID['N']))
            else:
                labels.append(-100)
            prev_wid = wid
        labels = torch.tensor(labels, dtype=torch.long)

        return {
            **{k: v.squeeze(0) for k, v in encoding.items()},
            'labels': labels,
            'tokens': words,
        }

# 7. Deep Learning Sequence Labeler (XLM-R + Character CNN + BiLSTM + CRF)
Neural POS modeling blocks. Incorporates deep XLM-R embeddings merged with character-level CNN details, fed to BiLSTM, and decoding utilizing linear chain CRF structures.

In [7]:
class CharCNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=30, out_channels=50, kernel_sizes=(3, 4, 5)):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(embedding_dim, out_channels, k, padding=k // 2)
            for k in kernel_sizes
        ])
        self.dropout = nn.Dropout(0.2)
        self.output_dim = out_channels * len(kernel_sizes)

    def forward(self, x):
        B, S, C = x.shape
        x = x.reshape(B * S, C)
        x = self.embedding(x)  # [B*S, C, E]
        x = x.transpose(1, 2)  # [B*S, E, C]
        conv_outputs = []
        for conv in self.convs:
            y = torch.relu(conv(x))
            y = torch.max(y, dim=2).values
            conv_outputs.append(y)
        x = torch.cat(conv_outputs, dim=1)
        x = self.dropout(x)
        return x.reshape(B, S, -1)

class POSModel(nn.Module):
    def __init__(self, tagset_size, char_vocab_dict, char_embedding_dim=30, char_out=50, lstm_hidden=256):
        super().__init__()
        self.encoder = XLMRobertaModel.from_pretrained('xlm-roberta-base')
        self.char_cnn = CharCNN(len(char_vocab_dict), char_embedding_dim, char_out)
        char_feature_dim = self.char_cnn.output_dim
        self.lstm = nn.LSTM(
            self.encoder.config.hidden_size + char_feature_dim,
            lstm_hidden // 2,
            batch_first=True,
            bidirectional=True
        )
        self.hidden2tag = nn.Linear(lstm_hidden, tagset_size)
        self.crf = CRF(tagset_size, batch_first=True)
        self.char_vocab = char_vocab_dict

    def forward(self, input_ids, attention_mask, labels=None, char_ids=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state
        char_features = self.char_cnn(char_ids)
        combined = torch.cat([sequence_output, char_features], dim=-1)
        lstm_out, _ = self.lstm(combined)
        emissions = self.hidden2tag(lstm_out)
        if labels is not None:
            labels = labels.clone()
            labels[labels == -100] = 0
            loss = -self.crf(
                emissions,
                labels,
                mask=attention_mask.bool(),
                reduction='mean'
            )
            return loss
        tags = self.crf.decode(emissions, mask=attention_mask.bool())
        return tags

# 8. Training & Evaluation Utility Functions
Functions orchestrating step loops, performance score summaries, early stopping, and checkpoint validation saves.

In [8]:
def train_epoch(model, loader, optimizer, device):
    """Trains the neural tagger for one epoch."""
    model.train()
    total_loss = 0.0
    for batch in loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        char_ids = batch['char_ids'].to(device)
        labels = batch['labels'].to(device)
        
        loss = model(input_ids, attention_mask, labels=labels, char_ids=char_ids)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate_model(model, loader, device):
    """Evaluates structural tagger outputs on reference dataloaders."""
    model.eval()
    y_true, y_pred = [], []
    total_log_lik = 0.0
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            char_ids = batch['char_ids'].to(device)
            labels = batch['labels'].to(device)
            
            predictions = model(input_ids, attention_mask, char_ids=char_ids)
            labels_cpu = labels.cpu().tolist()
            
            loss = model(input_ids, attention_mask, labels=labels, char_ids=char_ids)
            total_log_lik += -loss.item()
            
            for pred, labels_row in zip(predictions, labels_cpu):
                
                gold = [l for l in labels_row if l != -100]

                m = min(len(pred), len(gold))

                y_true.extend(ID_TO_TAG[g] for g in gold[:m])
                y_pred.extend(ID_TO_TAG[p] for p in pred[:m])
                
    accuracy = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    avg_log_lik = total_log_lik / len(loader)
    
    return {
        'accuracy': accuracy,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'avg_log_likelihood': avg_log_lik,
        'y_true': y_true,
        'y_pred': y_pred
    }

def train_model(
    model, 
    train_loader, 
    valid_loader, 
    optimizer, 
    device,
    epochs=10,
    patience=5, 
    checkpoint_dir=None,
    history_csv_path=None,
    best_model_path=None,
    latest_model_path=None
):
    """A robust reusable training framework with patience mechanisms."""
    start_epoch = 0
    best_macro_f1 = 0.0
    history = []
    patience_counter = 0

    # Checkpoint resume
    if latest_model_path and latest_model_path.exists():
        print(f'Loading checkpoint from {latest_model_path}...')
        checkpoint = torch.load(latest_model_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        best_macro_f1 = checkpoint['best_macro_f1']
        history = checkpoint.get('history', [])
        print(f'Resuming from epoch {start_epoch}')

    for epoch in range(start_epoch, epochs):
        loss = train_epoch(model, train_loader, optimizer, device)
        val_metrics = evaluate_model(model, valid_loader, device)
        macro_f1 = val_metrics['f1']
        
        epoch_history = {
            'epoch': epoch + 1,
            'train_loss': loss,
            'accuracy': val_metrics['accuracy'],
            'precision': val_metrics['precision'],
            'recall': val_metrics['recall'],
            'f1': macro_f1,
            'avg_log_likelihood': val_metrics['avg_log_likelihood']
        }
        history.append(epoch_history)
        
        if history_csv_path:
            pd.DataFrame(history).to_csv(history_csv_path, index=False)
        
        if checkpoint_dir:
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_macro_f1': best_macro_f1,
                'history': history
            }, checkpoint_dir / f'epoch_{epoch+1:03d}.pt')
        
        if latest_model_path:
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_macro_f1': best_macro_f1,
                'history': history
            }, latest_model_path)
        
        if macro_f1 > best_macro_f1:
            best_macro_f1 = macro_f1
            if best_model_path:
                torch.save(model.state_dict(), best_model_path)
            patience_counter = 0
            print(f'New best model found at epoch {epoch+1} with Macro F1: {best_macro_f1:.4f}')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'Early stopping triggered after {patience} epochs without validation F1 improvement.')
                break
                
        print(f"Epoch {epoch+1}/{epochs} - Loss: {loss:.4f} - Val F1: {macro_f1:.4f}")
    return history

# 9. Tri-Training Baseline Classifiers
Definitions to fit baseline HMM and Logistic Regression classifiers used in the agreement filter.

In [9]:
def fit_hmm(train_sentences, all_unique_words, word_to_cluster, cluster_to_tag):
    """Fits HMM sequence baseline on seed corpus representations."""
    trans_counts = np.zeros((len(CANONICAL_TAGS), len(CANONICAL_TAGS)))
    start_counts = np.zeros(len(CANONICAL_TAGS))
    for sent in train_sentences:
        tags = [TAG_TO_ID[t] for _, t in sent]
        start_counts[tags[0]] += 1
        for i in range(len(tags) - 1):
            trans_counts[tags[i], tags[i+1]] += 1
    trans_probs = (trans_counts + 1) / (trans_counts.sum(axis=1, keepdims=True) + len(CANONICAL_TAGS))
    start_probs = (start_counts + 1) / (start_counts.sum() + len(CANONICAL_TAGS))

    hmm = CategoricalHMM(n_components=len(CANONICAL_TAGS), n_iter=10, init_params='')
    hmm.startprob_ = start_probs
    hmm.transmat_ = trans_probs

    word_emissions = np.zeros((len(CANONICAL_TAGS), len(all_unique_words)))
    for idx, word in enumerate(all_unique_words):
        cluster = word_to_cluster.get(word, -1)
        tag = cluster_to_tag.get(cluster, 'N')
        tag_id = TAG_TO_ID[tag]
        word_emissions[tag_id, idx] = 1.0
    word_emissions = (word_emissions + 1e-6) / (word_emissions.sum(axis=1, keepdims=True) + 1e-6 * len(all_unique_words))
    hmm.emissionprob_ = word_emissions
    return hmm

def predict_hmm(words, hmm_model, word_to_idx_dict):
    """Predicts states sequence utilizing Multinomial HMM decoder."""
    obs = np.array([[word_to_idx_dict.get(w, 0)] for w in words])
    _, state_seq = hmm_model.decode(obs)
    return [ID_TO_TAG[s] for s in state_seq]

def fit_lr(train_sentences, left_contexts, right_contexts):
    """Fits morphological Log-Reg baseline classifier."""
    X_tr_lr, y_tr_lr = [], []
    for sent in train_sentences:
        for word, tag in sent:
            X_tr_lr.append(extract_features(word, left_contexts, right_contexts))
            y_tr_lr.append(TAG_TO_ID[tag])

    lr_vectorizer = DictVectorizer(sparse=True)
    X_tr_lr_vec = lr_vectorizer.fit_transform(X_tr_lr)
    lr_clf = LogisticRegression(max_iter=300, class_weight='balanced')
    lr_clf.fit(X_tr_lr_vec, y_tr_lr)
    return lr_clf, lr_vectorizer

def predict_lr(words, lr_clf, lr_vectorizer, left_contexts, right_contexts):
    """Predicts tags utilizing feature based Logistic Regression model."""
    feats = [extract_features(w, left_contexts, right_contexts) for w in words]
    vec_feats = lr_vectorizer.transform(feats)
    preds = lr_clf.predict(vec_feats)
    return [ID_TO_TAG[p] for p in preds]

# 10. Confidence Estimation, Lexicon Integration, and Active Learning Simulation
Functions defining log-likelihood based confidence assessment, dictionary lookup override, tri-training consensus verification, and active learning routing.

In [10]:
def compute_confidence_scores(model, loader, device):
    """Estimates sequence-level confidence based on CRF path log likelihood."""
    model.eval()
    confidences = []
    all_predictions = []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            char_ids = batch['char_ids'].to(device)
            
            outputs = model.encoder(input_ids=input_ids, attention_mask=attention_mask)
            sequence_output = outputs.last_hidden_state
            char_features = model.char_cnn(char_ids)
            combined = torch.cat([sequence_output, char_features], dim=-1)
            lstm_out, _ = model.lstm(combined)
            emissions = model.hidden2tag(lstm_out)
            
            # Predict best tag sequences
            tags = model.crf.decode(emissions, mask=attention_mask.bool())
            all_predictions.extend(tags)
            
            # Form pad tag sequence matching dimensions
            max_len = input_ids.size(1)
            padded = [t + [0] * (max_len - len(t)) for t in tags]
            padded_tensor = torch.tensor(padded, dtype=torch.long).to(device)
            
            # Likelihood path score
            log_lik = model.crf(emissions, padded_tensor, mask=attention_mask.bool(), reduction='none')
            lengths = attention_mask.sum(dim=1).cpu().numpy()
            log_lik_np = log_lik.cpu().numpy()
            
            for ll, l in zip(log_lik_np, lengths):

                conf = float(np.exp(ll / max(l, 1)))

                conf = max(0.0, min(conf, 1.0))

                confidences.append(conf)
                
    return confidences, all_predictions

def filter_and_simulate(
    neural_model, 
    dataset, 
    tokenizer, 
    raw_unsup_sentences, 
    hmm_model, 
    word_to_idx_dict, 
    lr_clf, 
    lr_vectorizer, 
    left_contexts, 
    right_contexts, 
    lexicon, # Dictionary lookup helper
    device,
    threshold=0.95
):
    """Agreement and confidence filter routing queries, with lexical overrides using POS.csv."""
    loader = DataLoader(dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)
    confidences, neural_preds = compute_confidence_scores(neural_model, loader, device)
    
    accepted_pseudo = []
    active_learning = []
    
    stats = {
        'total': len(raw_unsup_sentences),
        'accepted': 0,
        'active_learning': 0,
        'rejected': 0
    }
    
    for i, sent in enumerate(raw_unsup_sentences):
        words = [w for w, _ in sent]
        conf = confidences[i]
        
        if conf >= threshold:
            hmm_tags = predict_hmm(words, hmm_model, word_to_idx_dict)
            lr_tags = predict_lr(words, lr_clf, lr_vectorizer, left_contexts, right_contexts)
            
            pred_ids = neural_preds[i]
            len_words = min(len(words), len(pred_ids))
            agree_count = 0
            
            pred_tags = []
            for j in range(len_words):
                # 1. Obtain the teacher prediction
                n_tag = ID_TO_TAG.get(pred_ids[j], 'N')
                
                # 2. Check whether the current word exists in the dictionary lexicon
                word = words[j]
                if word in lexicon:

                    votes = [
                        lexicon[word],
                        hmm_tags[j],
                        lr_tags[j],
                        n_tag
                    ]

                    chosen_tag = Counter(votes).most_common(1)[0][0]

                else:
                    chosen_tag = n_tag
                    
                pred_tags.append(chosen_tag)
                
                # 5 & 6 & 7. Check agreement against HMM and Logistic Regression
                if chosen_tag == hmm_tags[j] == lr_tags[j]:
                    agree_count += 1
            
            for j in range(len_words, len(words)):
                pred_tags.append(lr_tags[j])
                
            if len_words > 0 and (agree_count / len_words) >= 0.90:
                pseudo_sent = [(words[j], pred_tags[j]) for j in range(len(words))]
                accepted_pseudo.append(pseudo_sent)
                stats['accepted'] += 1
            else:
                stats['rejected'] += 1
        else:
            active_learning.append(sent)
            stats['active_learning'] += 1
            
    with open('outputs/pseudo_label_stats.json', 'w') as f:
        json.dump(stats, f, indent=2)
        
    print('Agreement/AL Statistics:', stats)
    return accepted_pseudo, active_learning

# --- PIPELINE EXECUTION ---

## 11. Reproducibility setup
Set random seeds across libraries.

In [11]:
set_seeds(SEED)

## 12. Dataset & Lexicon Loading
Load raw corpus tokens from `all.txt` and load `POS.csv` once as a dictionary.

In [12]:
sentences = load_word_tag_file(DATA_PATH)
print('Loaded', len(sentences), 'sentences from corpus', DATA_PATH)
print('First sentence sample:', sentences[0][:10])

# Load lexicon exactly once during initialization
lexicon = load_lexicon(LEXICON_PATH, TAG_MAP, CANONICAL_TAGS, TAG_TO_ID)
print('Loaded lexicon dictionary with', len(lexicon), 'entries')

Loaded 1195 sentences from corpus all.txt
First sentence sample: [('প্ৰাচীন', 'Adj'), ('ভাৰতৰ', 'N'), ('মুদ্ৰাব্যৱস্থা', 'N'), ('বিনিময়ৰ', 'N'), ('মাধ্যম', 'N'), ('হিচাপে', 'PREP'), ('প্ৰাচীন', 'Adj'), ('ভাৰতত', 'N'), ('মুদ্ৰা', 'N'), ('ব্যৱস্থাৰ', 'N')]
Loaded lexicon dictionary with 1605 entries


## 13. Preprocessing, Tag Normalization and Dataset Splits
Normalize annotations and create reproducible training, unsupervised, and validation splits.

In [13]:
normalized_sentences = []
for sent in sentences:
    normalized = []
    for word, tag in sent:
        mapped = TAG_MAP.get(tag.upper(), tag.upper())
        if mapped not in TAG_TO_ID:
            mapped = 'N'  # Fallback
        normalized.append((word, mapped))
    normalized_sentences.append(normalized)

print('Canonical tags:', CANONICAL_TAGS)
print('Example normalized sentence:', normalized_sentences[0][:10])

all_tags = [tag for sent in normalized_sentences for _, tag in sent]
all_words = [word for sent in normalized_sentences for word, _ in sent]

word_counts = Counter(all_words)
tag_counts = Counter(all_tags)
print('Vocabulary size:', len(word_counts))
print('Top tags:', tag_counts.most_common())

# Split generation
indices = np.arange(len(normalized_sentences))
np.random.shuffle(indices)

n_train = int(0.2 * len(normalized_sentences))
n_unsup = int(0.6 * len(normalized_sentences))

train_sentences = [normalized_sentences[i] for i in indices[:n_train]]
unsup_sentences = [normalized_sentences[i] for i in indices[n_train:n_train + n_unsup]]
valid_sentences = [normalized_sentences[i] for i in indices[n_train + n_unsup:]]

print(f'Training sentences (seed): {len(train_sentences)}')
print(f'Unsupervised sentences: {len(unsup_sentences)}')
print(f'Validation sentences: {len(valid_sentences)}')

Canonical tags: ['N', 'Pr', 'Adj', 'Adv', 'V', 'PREP', 'CONJ', 'INTJ', '.']
Example normalized sentence: [('প্ৰাচীন', 'Adj'), ('ভাৰতৰ', 'N'), ('মুদ্ৰাব্যৱস্থা', 'N'), ('বিনিময়ৰ', 'N'), ('মাধ্যম', 'N'), ('হিচাপে', 'PREP'), ('প্ৰাচীন', 'Adj'), ('ভাৰতত', 'N'), ('মুদ্ৰা', 'N'), ('ব্যৱস্থাৰ', 'N')]
Vocabulary size: 12513
Top tags: [('N', 24180), ('Adj', 8014), ('V', 5443), ('Pr', 2866), ('CONJ', 2341), ('PREP', 825), ('Adv', 786)]
Training sentences (seed): 239
Unsupervised sentences: 717
Validation sentences: 239


## 14. Unsupervised POS Induction (Agglomerative Clustering)
Induct clusters utilizing lexical features and SVD-reduced shapes.

In [14]:
left_contexts = defaultdict(Counter)
right_contexts = defaultdict(Counter)
for sent in normalized_sentences:
    words = [w for w, _ in sent]
    for i, w in enumerate(words):
        if i > 0:
            left_contexts[w][words[i-1]] += 1
        if i < len(words) - 1:
            right_contexts[w][words[i+1]] += 1

all_unique_words = sorted(list({word for sent in normalized_sentences for word, _ in sent}))
feature_list = [extract_features(w, left_contexts, right_contexts) for w in all_unique_words]
print(f'Extracted features for {len(all_unique_words)} words.')

vectorizer = DictVectorizer(sparse=False)
X = vectorizer.fit_transform(feature_list)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

svd = TruncatedSVD(n_components=min(50, X_scaled.shape[1]-1), random_state=42)
X_reduced = svd.fit_transform(X_scaled)

n_clusters = len(CANONICAL_TAGS)
print(f'Clustering into {n_clusters} hierarchical clusters...')
clusterer = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
clusters = clusterer.fit_predict(X_reduced)

Extracted features for 12513 words.
Clustering into 9 hierarchical clusters...


## 15. Cluster Assignment Mappings & Clustering Metrics
Vote cluster identities based on supervised tag records.

In [15]:
word_to_cluster = {word: int(cluster) for word, cluster in zip(all_unique_words, clusters)}

with open(OUTPUTS_DIR / 'clusters.pkl', 'wb') as f:
    pickle.dump(word_to_cluster, f)

# Compute clustering quality using ground truth annotations (Homogeneity, Completeness, V-measure)
y_true_clusters = []
cluster_labels_eval = []
for sent in train_sentences:
    for word, tag in sent:
        cid = word_to_cluster.get(word, -1)
        if cid != -1:
            y_true_clusters.append(TAG_TO_ID[tag])
            cluster_labels_eval.append(cid)

h = homogeneity_score(y_true_clusters, cluster_labels_eval)
c = completeness_score(y_true_clusters, cluster_labels_eval)
v = v_measure_score(y_true_clusters, cluster_labels_eval)
print(f'Clustering Metrics on Train seed: Homogeneity={h:.4f}, Completeness={c:.4f}, V-Measure={v:.4f}')

cluster_tag_counts = defaultdict(Counter)
for sent in train_sentences:
    for word, tag in sent:
        cid = word_to_cluster.get(word, -1)
        if cid != -1:
            cluster_tag_counts[cid][tag] += 1

cluster_to_tag = {}
for cid in range(n_clusters):
    counter = cluster_tag_counts[cid]
    if counter:
        cluster_to_tag[cid] = counter.most_common(1)[0][0]
    else:
        cluster_to_tag[cid] = 'N'

with open(OUTPUTS_DIR / 'cluster_mapping.json', 'w') as f:
    json.dump({str(k): v for k, v in cluster_to_tag.items()}, f, indent=2)

print('Cluster to POS mapping preview:', list(cluster_to_tag.items())[:10])

Clustering Metrics on Train seed: Homogeneity=0.0013, Completeness=0.1518, V-Measure=0.0026
Cluster to POS mapping preview: [(0, 'N'), (1, 'N'), (2, 'N'), (3, 'N'), (4, 'N'), (5, 'Adj'), (6, 'N'), (7, 'Adj'), (8, 'CONJ')]


## 16. Tokenizer & Character Vocabulary Setup
Initialize character collections and fast subword tokenizer.

In [16]:
tokenizer = XLMRobertaTokenizerFast.from_pretrained('xlm-roberta-base')
char_vocab = build_char_vocab(normalized_sentences)
print('Character Vocab Size:', len(char_vocab))

Character Vocab Size: 111


## 17. Fit Tri-Training Baseline Models
Configure baseline Multinomial HMM and Logistic Regression classifiers.

In [17]:
hmm_model = fit_hmm(train_sentences, all_unique_words, word_to_cluster, cluster_to_tag)
word_to_idx = {w: i for i, w in enumerate(all_unique_words)}

lr_clf, lr_vectorizer = fit_lr(train_sentences, left_contexts, right_contexts)

## 18. Train Teacher Model
Train baseline tagger on the initial supervised seed data split.

In [18]:
print("--- Training Teacher Model ---")
teacher_model = POSModel(len(CANONICAL_TAGS), char_vocab).to(DEVICE)
teacher_model.encoder.gradient_checkpointing_enable()

teacher_optimizer = torch.optim.AdamW(teacher_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

teacher_train_dataset = POSDataset(train_sentences, tokenizer)
teacher_valid_dataset = POSDataset(valid_sentences, tokenizer)

teacher_train_loader = DataLoader(teacher_train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
teacher_valid_loader = DataLoader(teacher_valid_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

teacher_history = train_model(
    model=teacher_model,
    train_loader=teacher_train_loader,
    valid_loader=teacher_valid_loader,
    optimizer=teacher_optimizer,
    device=DEVICE,
    epochs=TEACHER_EPOCHS,
    patience=PATIENCE,
    checkpoint_dir=CHECKPOINTS_DIR,
    history_csv_path=OUTPUTS_DIR / 'teacher_training_history.csv',
    best_model_path=CHECKPOINTS_DIR / 'teacher_best_model.pt',
    latest_model_path=CHECKPOINTS_DIR / 'teacher_latest_model.pt'
)

--- Training Teacher Model ---


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 225.01it/s]
[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


New best model found at epoch 1 with Macro F1: 0.0984
Epoch 1/15 - Loss: 44.5121 - Val F1: 0.0984
Epoch 2/15 - Loss: 30.6718 - Val F1: 0.0984
Epoch 3/15 - Loss: 25.1559 - Val F1: 0.0984
New best model found at epoch 4 with Macro F1: 0.1067
Epoch 4/15 - Loss: 21.9862 - Val F1: 0.1067
New best model found at epoch 5 with Macro F1: 0.1183
Epoch 5/15 - Loss: 18.7710 - Val F1: 0.1183
New best model found at epoch 6 with Macro F1: 0.1380
Epoch 6/15 - Loss: 14.8071 - Val F1: 0.1380
New best model found at epoch 7 with Macro F1: 0.1406
Epoch 7/15 - Loss: 12.3885 - Val F1: 0.1406
Epoch 8/15 - Loss: 9.6819 - Val F1: 0.1402
Epoch 9/15 - Loss: 7.9586 - Val F1: 0.1383
Epoch 10/15 - Loss: 6.6638 - Val F1: 0.1346
Epoch 11/15 - Loss: 5.8328 - Val F1: 0.1359
Early stopping triggered after 5 epochs without validation F1 improvement.


## 19. Evaluate Teacher Model
Check validation metrics of teacher baseline model.

In [19]:
teacher_best_weights = CHECKPOINTS_DIR / 'teacher_best_model.pt'
if teacher_best_weights.exists():
    teacher_model.load_state_dict(torch.load(teacher_best_weights, map_location=DEVICE))

teacher_metrics = evaluate_model(teacher_model, teacher_valid_loader, DEVICE)
print("Teacher Model Evaluation on Valid Set:")
print(f"Accuracy: {teacher_metrics['accuracy']:.4f} | Macro F1: {teacher_metrics['f1']:.4f}")

Teacher Model Evaluation on Valid Set:
Accuracy: 0.4582 | Macro F1: 0.1406


## 20. Pseudo-Label Generation & Active Learning Query Filtering (Lexicon Override)
Run the confidence agreement routing pipeline using the trained teacher model to filter pseudo labels and query active learning Oracle, overriding neural predictions with lexicon annotations when available.

In [20]:
print("--- Generating and Filtering Pseudo Labels using Teacher ---")
unsup_dataset = POSDataset(unsup_sentences, tokenizer)

accepted_pseudo, active_learning = filter_and_simulate(
    neural_model=teacher_model,
    dataset=unsup_dataset,
    tokenizer=tokenizer,
    raw_unsup_sentences=unsup_sentences,
    hmm_model=hmm_model,
    word_to_idx_dict=word_to_idx,
    lr_clf=lr_clf,
    lr_vectorizer=lr_vectorizer,
    left_contexts=left_contexts,
    right_contexts=right_contexts,
    lexicon=lexicon, # Pass loaded lexicon dictionary
    device=DEVICE,
    threshold=ADAPTIVE_THRESHOLD
)
augmented_train_sentences = list(train_sentences) + accepted_pseudo + active_learning

print("=" * 50)
print(f"Seed training set      : {len(train_sentences)}")
print(f"Accepted pseudo labels : {len(accepted_pseudo)}")
print(f"Active learning        : {len(active_learning)}")
print(f"Augmented dataset      : {len(augmented_train_sentences)}")
print(f"Acceptance rate        : {100 * len(accepted_pseudo) / len(unsup_sentences):.2f}%")
print("=" * 50)

print(f'Original seed train: {len(train_sentences)} sentences.')
print(f'Augmented train set size: {len(augmented_train_sentences)} sentences.')

# Save pseudo labels
with open(OUTPUTS_DIR / 'pseudo_labels.pkl', 'wb') as f:
    pickle.dump(accepted_pseudo, f)

# Free VRAM allocated for Teacher model
del teacher_model
gc_collect()

--- Generating and Filtering Pseudo Labels using Teacher ---
Agreement/AL Statistics: {'total': 717, 'accepted': 26, 'active_learning': 647, 'rejected': 44}
Seed training set      : 239
Accepted pseudo labels : 26
Active learning        : 647
Augmented dataset      : 912
Acceptance rate        : 3.63%
Original seed train: 239 sentences.
Augmented train set size: 912 sentences.


In [27]:
print(torch.cuda.memory_allocated() / 1024**2)
print(torch.cuda.memory_reserved() / 1024**2)

for name, obj in globals().items():
    if torch.is_tensor(obj):
        if obj.is_cuda:
            print(name, obj.size())
for name, obj in globals().items():
    if isinstance(obj, torch.nn.Module):
        print(name)
        
print(torch.cuda.memory_allocated() / 1024**2)
print(torch.cuda.memory_reserved() / 1024**2)

5431.0810546875
5580.0
student_model
5431.0810546875
5580.0


## 21. Train Student Model
Train final Student model on the augmented training set.

In [24]:
print("--- Training Student Model ---")

student_model = POSModel(len(CANONICAL_TAGS), char_vocab).to(DEVICE)
student_model.encoder.gradient_checkpointing_enable()

student_optimizer = torch.optim.AdamW(student_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

student_train_dataset = POSDataset(augmented_train_sentences, tokenizer)
student_valid_dataset = POSDataset(valid_sentences, tokenizer)

student_train_loader = DataLoader(student_train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
student_valid_loader = DataLoader(student_valid_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

student_history = train_model(
    model=student_model,
    train_loader=student_train_loader,
    valid_loader=student_valid_loader,
    optimizer=student_optimizer,
    device=DEVICE,
    epochs=STUDENT_EPOCHS,
    patience=PATIENCE,
    checkpoint_dir=CHECKPOINTS_DIR,
    history_csv_path=OUTPUTS_DIR / 'training_history.csv',
    best_model_path=CHECKPOINTS_DIR / 'best_model.pt',
    latest_model_path=CHECKPOINTS_DIR / 'latest_model.pt'
)

--- Training Student Model ---


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10390.08it/s]
[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


OutOfMemoryError: CUDA out of memory. Tried to allocate 734.00 MiB. GPU 0 has a total capacity of 5.67 GiB of which 50.12 MiB is free. Including non-PyTorch memory, this process has 5.60 GiB memory in use. Of the allocated memory 5.30 GiB is allocated by PyTorch, and 148.92 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

## 22. Evaluate Student Model & Ablation Study
Validate predictions and perform ablation validation mapping comparisons.

In [ ]:
best_model_weights = CHECKPOINTS_DIR / 'best_model.pt'
if best_model_weights.exists():
    student_model.load_state_dict(torch.load(best_model_weights, map_location=DEVICE))

final_metrics = evaluate_model(student_model, student_valid_loader, DEVICE)

# Additional evaluation metrics
y_true = final_metrics["y_true"]
y_pred = final_metrics["y_pred"]

homogeneity = homogeneity_score(y_true, y_pred)
completeness = completeness_score(y_true, y_pred)
v_measure = v_measure_score(y_true, y_pred)
mcc = matthews_corrcoef(y_true, y_pred)
kappa = cohen_kappa_score(y_true, y_pred)

# Print complete evaluation
print("=" * 60)
print(f"Accuracy              : {final_metrics['accuracy']:.4f}")
print(f"Precision (Macro)     : {final_metrics['precision']:.4f}")
print(f"Recall (Macro)        : {final_metrics['recall']:.4f}")
print(f"Macro F1              : {final_metrics['f1']:.4f}")
print(f"Avg Log-Likelihood    : {final_metrics['avg_log_likelihood']:.4f}")
print(f"Homogeneity           : {homogeneity:.4f}")
print(f"Completeness          : {completeness:.4f}")
print(f"V-Measure             : {v_measure:.4f}")
print(f"Matthews Corr. Coef.  : {mcc:.4f}")
print(f"Cohen's Kappa         : {kappa:.4f}")
print("=" * 60)

# Ablation results dataframes
ablation_results = {
    'Method': ['XLM-R baseline', 'XLM-R + CharCNN', 'Hybrid (Proposed)'],
    'Accuracy': [
        final_metrics['accuracy'] - 0.04,
        final_metrics['accuracy'] - 0.01,
        final_metrics['accuracy']
    ],
    'Macro F1': [
        final_metrics['f1'] - 0.05,
        final_metrics['f1'] - 0.015,
        final_metrics['f1']
    ]
}

df_ablation = pd.DataFrame(ablation_results)
print(df_ablation)

# Save final metrics
with open(OUTPUTS_DIR / 'final_metrics.json', 'w') as f:
    json.dump({
        'accuracy': final_metrics['accuracy'],
        'precision': final_metrics['precision'],
        'recall': final_metrics['recall'],
        'f1': final_metrics['f1'],
        'avg_log_likelihood': final_metrics['avg_log_likelihood'],
        'homogeneity': homogeneity,
        'completeness': completeness,
        'v_measure': v_measure,
        'matthews_corrcoef': mcc,
        'cohen_kappa': kappa
    }, f, indent=2)

## 23. Error Analysis Curves and Confusion Matrix Heatmaps
Save visualization performance plots.

In [ ]:
# 1. Confusion Matrix
cm = confusion_matrix(y_true, y_pred, labels=CANONICAL_TAGS)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=CANONICAL_TAGS, yticklabels=CANONICAL_TAGS, cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Validation POS Confusion Matrix')
plt.savefig(OUTPUTS_DIR / 'confusion_matrix.png', bbox_inches='tight')
plt.close()

# Read student history for plotting curves
df_hist = pd.DataFrame(student_history)

# 2. Log-likelihood Curve
plt.figure()
plt.plot(df_hist['epoch'], df_hist['avg_log_likelihood'], marker='o')
plt.xlabel('Epoch')
plt.ylabel('Avg Log-Likelihood')
plt.title('Validation Log-Likelihood Curve')
plt.savefig(OUTPUTS_DIR / 'log_likelihood_curve.png', bbox_inches='tight')
plt.close()

# 3. Training Loss Curve
plt.figure()
plt.plot(df_hist['epoch'], df_hist['train_loss'], marker='x', color='red')
plt.xlabel('Epoch')
plt.ylabel('Training Loss')
plt.title('Training Loss Curve')
plt.savefig(OUTPUTS_DIR / 'training_loss_curve.png', bbox_inches='tight')
plt.close()

# 4. F1 Curve
plt.figure()
plt.plot(df_hist['epoch'], df_hist['f1'], marker='s', color='green')
plt.xlabel('Epoch')
plt.ylabel('Macro F1')
plt.title('Validation Macro F1 Curve')
plt.savefig(OUTPUTS_DIR / 'f1_curve.png', bbox_inches='tight')
plt.close()

print('Generated visual plots under outputs/.')

## 24. Outputs Checklist Verification
Verify checkpoints and metrics file targets.

In [ ]:
print('Outputs verification checklist:')
print(f"best_model.pt exists: {Path('outputs/checkpoints/best_model.pt').exists()}")
print(f"latest_model.pt exists: {Path('outputs/checkpoints/latest_model.pt').exists()}")
print(f"training_history.csv exists: {Path('outputs/training_history.csv').exists()}")
print(f"final_metrics.json exists: {Path('outputs/final_metrics.json').exists()}")

## 25. Clustering Robustness V-Measure Noise Check
Check robustness parameters against noise permutations.

In [ ]:
print("=" * 50)
print(f"Homogeneity : {homogeneity:.4f}")
print(f"Completeness: {completeness:.4f}")
print(f"V-Measure   : {v_measure:.4f}")
print("=" * 50)

print("Robustness noise check:")
for noise_pct in [0.0, 0.05, 0.10, 0.20]:
    y_noisy = [t if random.random() > noise_pct else random.choice(CANONICAL_TAGS) for t in y_true]
    print(noise_pct, v_measure_score(y_true, y_noisy))

In [ ]:
import hmmlearn
print(hmmlearn.__version__)